In [1]:
import sys
import os
import pandas as pd
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.data_loading import consumers, accounts, transactions, category_mapping
from scripts.backfill_transactions import build_backfill_df
from scripts.feature_creation import create_all_features, print_feature_summary, print_feature_groups

df = build_backfill_df()
df.head()

Removed 2,012 duplicate transaction rows (from 6,407,321 to 6,405,309).


,prism_consumer_id,date,balance,credit_or_debit,amount_change,DQ_TARGET
0,3023,2021-08-31,225.95,starting value,0.00,0.0
1,3023,2021-03-24,205.32,DEBIT,20.63,0.0
2,3023,2021-03-27,445.32,CREDIT,240.00,0.0
3,3023,2021-03-27,785.32,CREDIT,340.00,0.0
4,3023,2021-03-29,760.32,DEBIT,25.00,0.0


## Feature Creation

All feature creation logic has been moved to `scripts/feature_creation.py`. This includes:

**Core Features:**
- Balance features (all-time and time windows)
- Daily cashflow features (30d, 60d, 90d, 180d)
- Transaction features
- Category features (top 30 categories, all-time and 90d)
- Grouped category features (income, essentials, discretionary)

**Risk Indicators (NEW):**
- **Overdraft & fees**: Overdraft fees, account fees (all-time, 30d, 90d)
- **Low balance risk**: Days below zero/50/100, consecutive negative days, zero crossings
- **Income regularity**: Income frequency, consistency, paycheck patterns
- **Paycheck-to-paycheck**: Min balance before income, depletion rate, days to deplete half balance

In [2]:
# Create all features using the feature_creation module
features_df = create_all_features(df, transactions, category_mapping)

# Display first few rows
features_df.head()

FEATURE CREATION PIPELINE
Preparing daily data...
Daily data shape: (1183217, 6)
Creating balance features...
Balance features shape: (12900, 9)
Creating daily window features...
  30d: (12900, 9)
  60d: (12900, 9)
  90d: (12900, 9)
  180d: (12900, 9)
Creating transaction features...
Transaction features shape: (12900, 7)
Creating category features (top 30 categories)...
Category features (all-time): (14491, 60)
Category features (90d): (14491, 60)
Creating grouped category features...
Grouped category features shape: (14305, 5)
Creating overdraft & fee features...
Overdraft & fee features shape: (7219, 15)
Creating low balance risk features...
Low balance risk features shape: (12900, 15)
Creating income regularity features...
Income regularity features shape: (14035, 7)
Creating paycheck-to-paycheck features...
Paycheck-to-paycheck features shape: (12900, 3)

Joining all features...

FINAL FEATURE MATRIX
Shape: (12900, 218)
Number of features: 217
Number of consumers: 12900


,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,balance__mean__30d,...,income__coefficient_of_variation,income__avg_days_between__90d,income__count__90d,paycheck__has_regular,paycheck__consistency_score,paycheck__count__90d,balance__min_before_income__avg__90d,balance__depletion_rate__90d,days_to_deplete_half_balance__avg__90d,DQ_TARGET
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,276.961538,70.09,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,-497.176071,...,0.672316,14.166667,7.0,0.0,0.686953,5.0,-853.385000,89.173235,9.333333,0.0
1,1674.533585,1758.35,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,2671.932727,...,0.496148,8.400000,11.0,1.0,0.721294,8.0,2507.552222,126.456250,0.000000,0.0
10,-106.435115,-98.40,-1108.49,929.25,501.726601,0.595420,0.633588,0.839695,131.0,-382.525455,...,0.780098,7.000000,13.0,1.0,0.747066,8.0,-643.487000,155.752500,6.000000,0.0
100,-3231.228909,-3752.93,-6273.18,802.40,2080.213280,0.963636,0.963636,0.981818,55.0,-4166.195000,...,0.824226,4.875000,17.0,0.0,0.708656,14.0,-4985.593333,465.608462,0.000000,0.0
1000,1013.427875,615.39,-22.85,12589.57,1545.044777,0.025000,0.112500,0.437500,80.0,601.327692,...,0.393328,5.600000,16.0,0.0,0.859934,13.0,313.198462,400.970455,1.875000,0.0


In [3]:
# Save features for later use
features_df.to_csv('../output/features.csv', index=True)

## Feature Analysis

In [4]:
# Print feature summary statistics
summary = print_feature_summary(features_df)

# Also return the summary dataframe for further analysis
summary.head(30)


Top 30 features by non-zero rate:
                       feature  nonzero_rate          mean           std
44                n_days__180d      1.000000  8.225372e+01  4.682480e+01
45                  tx__n__all      1.000000  4.039748e+02  3.676333e+02
26                 n_days__60d      1.000000  3.567775e+01  1.520348e+01
25                   n_tx__60d      1.000000  1.659029e+02  1.358372e+02
34                   n_tx__90d      1.000000  2.415008e+02  1.970092e+02
35                 n_days__90d      1.000000  5.210047e+01  2.273709e+01
17                 n_days__30d      1.000000  1.864915e+01  7.663812e+00
16                   n_tx__30d      1.000000  8.645163e+01  7.113610e+01
8                  n_days__all      1.000000  9.172225e+01  5.621988e+01
43                  n_tx__180d      1.000000  3.584177e+02  2.944847e+02
0           balance__mean__all      0.997907  4.143456e+03  2.843717e+04
1         balance__median__all      0.997597  3.733590e+03  2.771031e+04
36         balan

,feature,nonzero_rate,mean,std
44,n_days__180d,1.000000,8.225372e+01,4.682480e+01
45,tx__n__all,1.000000,4.039748e+02,3.676333e+02
26,n_days__60d,1.000000,3.567775e+01,1.520348e+01
25,n_tx__60d,1.000000,1.659029e+02,1.358372e+02
34,n_tx__90d,1.000000,2.415008e+02,1.970092e+02
35,n_days__90d,1.000000,5.210047e+01,2.273709e+01
17,n_days__30d,1.000000,1.864915e+01,7.663812e+00
16,n_tx__30d,1.000000,8.645163e+01,7.113610e+01
8,n_days__all,1.000000,9.172225e+01,5.621988e+01
43,n_tx__180d,1.000000,3.584177e+02,2.944847e+02


In [5]:
# Print features organized by groups
print_feature_groups(features_df)


FEATURE GROUPS

Balance behavior (40 features):
  - balance__mean__all
  - balance__median__all
  - balance__min__all
  - balance__max__all
  - balance__std__all
  - balance__pct_negative__all
  - balance__pct_below_100__all
  - balance__pct_below_500__all
  - balance__mean__30d
  - balance__min__30d
  - balance__std__30d
  - balance__pct_negative__30d
  - balance__mean__60d
  - balance__min__60d
  - balance__std__60d
  - balance__pct_negative__60d
  - balance__mean__90d
  - balance__min__90d
  - balance__std__90d
  - balance__pct_negative__90d
  - balance__mean__180d
  - balance__min__180d
  - balance__std__180d
  - balance__pct_negative__180d
  - balance__days_below_zero__all
  - balance__days_below_50__all
  - balance__days_below_100__all
  - balance__consecutive_negative_days_max__all
  - balance__zero_crossings_count__all
  - balance__days_below_zero__30d
  - balance__days_below_50__30d
  - balance__days_below_100__30d
  - balance__consecutive_negative_days_max__30d
  - balance__

## Model Evaluation

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Prepare data
eval_df = features_df.replace([np.inf, -np.inf], np.nan)
eval_df = eval_df[eval_df["DQ_TARGET"].notna()].copy()

y = eval_df["DQ_TARGET"].astype(int)
X = (
    eval_df.drop(columns=["DQ_TARGET"])
      .select_dtypes(include=[np.number])
      .fillna(0)
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Tree-based gradient boosting (handles nonlinearity)
model = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=300,
    learning_rate=0.05,
    min_samples_leaf=50,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
val_prob = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_prob)

print("HistGradientBoosting AUC:", round(auc, 4))

HistGradientBoosting AUC: 0.7985
